# Introduction

This notebook demonstrates the initial stages of the single-pulse analysis: detrending and the calculation of fundamental statistical quantities (energy and equivalent width). It uses simulated data but is otherwise identical to the procedure used in the paper.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.optimize as optimize

np.random.seed(0)

In [ ]:
def gaussian(t, t0, amp, sig):
    """Gaussian pulse"""
    return amp*np.exp(-(t - t0)**2/(2*sig**2))

def gaussian_with_offset(t, t0, amp, sig, c):
    """Gaussian pulse with vertical offset"""
    return gaussian(t, t0, amp, sig) + c

def line(t, m, b):
    """Line for detrending"""
    return m*t + b

In [ ]:
# time series parameters
t_samp = 40.96 * 10**(-6)   # sample time (ms)
N = 1024                    # number of samples in time series
L = N*t_samp                # length of chunk of time series (ms)
ts = np.arange(0, N)*t_samp # times
ts = ts - L/2

# noise parameters
mu = 0
std = 1

# pulse properties
amp = 10
sig = 20*t_samp

# generate simulated data
pulse = gaussian(ts, np.mean(ts), amp, sig)
noise = np.random.normal(loc=mu, scale=std, size=N)
signal = pulse + noise

plt.plot(ts, signal, 'k')
plt.xlabel('Time (ms)')
plt.ylabel('Intensity (arbitrary units)')
plt.show()

In [ ]:
def analysis(ts, signal, f=5):
    """Detrends signal and extracts pulse energy and equivalent width"""

    # 1) fit Gaussian with offset to pulse
    popt, pcov = optimize.curve_fit(gaussian_with_offset, ts, signal, p0=[0, np.max(signal), 20*t_samp, 0],
                                    bounds=((np.min(ts), 0, t_samp, np.min(ts)),
                                            (np.max(ts), np.max(signal), np.max(ts) - np.min(ts), np.max(ts))))
    t0_b, amp_b, sig_b, c_b = popt

    # 2) make on-pulse mask
    on_pulse_mask = (np.abs(ts - t0_b) <= f*sig_b)

    # 3) fit line to off-pulse region
    popt, pcov = optimize.curve_fit(line, ts[~on_pulse_mask], signal[~on_pulse_mask], p0=[0, 0])
    m_b, b_b = popt

    # 4) detrend entire pulse
    detrended_signal = signal - line(ts, m_b, b_b)

    # 5) re-fit Gaussian (without offset) to pulse
    popt, pcov = optimize.curve_fit(gaussian, ts, detrended_signal, p0=[0, np.max(detrended_signal), 20*t_samp],
                                    bounds=((np.min(ts), 0, t_samp),
                                            (np.max(ts), 2*np.max(signal), np.max(ts) - np.min(ts))))
    t0_b, amp_b, sig_b, = popt

    # 6) make new on-pulse mask
    on_pulse_mask = (np.abs(ts - t0_b <= f*sig_b))

    # 7) calculate energy and equivalent width. since most pulses are faint,
    #    we use the fitted amplitude of the pulse as the peak flux density,
    #    rather than the actual peak intensity of the time series in the
    #    on-pulse region; using the actual intensity leads to extreme
    #    fluctuations due to noise.
    energy = np.sum(detrended_signal[on_pulse_mask])*t_samp
    W_eq = energy/amp_b

    plt.plot(ts, detrended_signal, 'k', label='Detrended pulse')
    plt.plot(ts, gaussian(ts, t0_b, amp_b, sig_b), label='Detrended fit')
    plt.xlabel('Time (ms)')
    plt.ylabel('Intensity (arbitrary units)')
    plt.legend()
    plt.show()
    
    return energy, W_eq

energy, W_eq = analysis(ts, signal)